In [1]:
from pyspark import SparkContext
import datetime

In [2]:
sc = SparkContext.getOrCreate()
sc.setLogLevel("ERROR")

with open("movies.txt", "r", encoding="utf-8") as f:
    movies_lines = [line.strip() for line in f.readlines()]
movies_rdd = sc.parallelize(movies_lines)
movies_parsed = movies_rdd.map(lambda line: line.split(",")) \
                         .map(lambda x: (int(x[0]), (x[1], x[2])))

def load_ratings(filename):
    with open(filename, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]
    return sc.parallelize(lines).map(lambda line: line.split(",")) \
             .map(lambda x: (int(x[0]), int(x[1]), float(x[2]), int(x[3])))
ratings1_rdd = load_ratings("ratings_1.txt")
ratings2_rdd = load_ratings("ratings_2.txt")
ratings_rdd = ratings1_rdd.union(ratings2_rdd)

with open("users.txt", "r", encoding="utf-8") as f:
    users_lines = [line.strip() for line in f.readlines()]
users_rdd = sc.parallelize(users_lines)
users_parsed = users_rdd.map(lambda line: line.split(",")) \
                        .map(lambda x: (int(x[0]), x[1], int(x[2]), int(x[3]), x[4]))

with open("occupation.txt", "r", encoding="utf-8") as f:
    occ_lines = [line.strip() for line in f.readlines()]
occupation_rdd = sc.parallelize(occ_lines)
occupation_parsed = occupation_rdd.map(lambda line: line.split(",")) \
                                  .map(lambda x: (int(x[0]), x[1]))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 22:53:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Section 01

In [3]:
# Tạo map movie ID -> title
movie_titles = movies_parsed.map(lambda x: (x[0], x[1][0]))

# Tính tổng rating và số lượt theo movie
ratings_movie = ratings_rdd.map(lambda x: (x[1], (x[2], 1)))
ratings_sum_count = ratings_movie.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Tính điểm trung bình, chỉ giữ phim có >=50 lượt (ở đây dùng 5 do dữ liệu nhỏ)
avg_ratings = ratings_sum_count.mapValues(lambda x: (x[0] / x[1], x[1])).filter(lambda x: x[1][1] >= 5)

# Tìm phim có điểm cao nhất
max_movie = avg_ratings.map(lambda x: (x[1][0], x[0])).max()
max_title = movie_titles.lookup(max_movie[1])[0]
print(f"Phim cao nhất: {max_title} - {max_movie[0]:.2f}")

Phim cao nhất: Sunset Boulevard (1950) - 4.36


## Section 02

In [4]:
# Gắn thể loại cho mỗi rating
movie_genres = movies_parsed.map(lambda x: (x[0], x[1][1].split("|")))
ratings_with_genre = ratings_rdd.map(lambda x: (x[1], x[2])).join(movie_genres)
genre_rating = ratings_with_genre.flatMap(lambda x: [(g, x[1][0]) for g in x[1][1]])

# Tính trung bình theo thể loại
genre_sum_count = genre_rating.mapValues(lambda r: (r, 1)).reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
genre_avg = genre_sum_count.mapValues(lambda x: x[0]/x[1])
print("Điểm TB theo thể loại:", genre_avg.collect())

Điểm TB theo thể loại: [('Biography', 3.56), ('Drama', 3.7578125), ('Family', 3.6666666666666665), ('Thriller', 3.7037037037037037), ('Adventure', 3.6325301204819276), ('Film-Noir', 4.357142857142857), ('Crime', 3.8095238095238093), ('Sci-Fi', 3.7314814814814814), ('Horror', 4.0), ('Mystery', 4.0), ('Fantasy', 3.8620689655172415), ('Action', 3.712962962962963)]


## Bài 03

In [5]:
# Thêm giới tính user vào rating
user_gender = users_parsed.map(lambda x: (x[0], x[1]))
ratings_user = ratings_rdd.map(lambda x: (x[0], (x[1], x[2])))
joined = ratings_user.join(user_gender)
movie_gender_rating = joined.map(lambda x: ((x[1][0][0], x[1][1]), x[1][0][1]))

# Tính trung bình theo (movie, gender)
avg = movie_gender_rating.mapValues(lambda r: (r,1)).reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
avg = avg.mapValues(lambda x: x[0]/x[1])
print("Điểm TB theo phim và giới tính:", avg.collect()[:10])

Điểm TB theo phim và giới tính: [((1012, 'F'), 4.0), ((1039, 'F'), 3.9), ((1010, 'M'), 3.55), ((1013, 'F'), 3.9375), ((1015, 'M'), 4.333333333333333), ((1047, 'F'), 3.6666666666666665), ((1050, 'F'), 3.3214285714285716), ((1037, 'F'), 3.8), ((1030, 'F'), 3.0), ((1030, 'M'), 3.3333333333333335)]


## Bài 04

In [6]:
# Phân nhóm tuổi
def age_group(age):
    return "18-24" if age<25 else "25-34" if age<35 else "35-44" if age<45 else "45-54" if age<55 else "55+"
user_age = users_parsed.map(lambda x: (x[0], age_group(x[2])))

# Join và tính trung bình theo (movie, age group)
ratings_user = ratings_rdd.map(lambda x: (x[0], (x[1], x[2])))
joined = ratings_user.join(user_age)
movie_age_rating = joined.map(lambda x: ((x[1][0][0], x[1][1]), x[1][0][1]))
sumcnt = movie_age_rating.mapValues(lambda r: (r,1)).reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
avg = sumcnt.mapValues(lambda x: x[0]/x[1])
print("Điểm TB theo nhóm tuổi:", avg.collect()[:10])

Điểm TB theo nhóm tuổi: [((1050, '35-44'), 3.642857142857143), ((1043, '35-44'), 3.75), ((1013, '18-24'), 4.0), ((1030, '45-54'), 3.3333333333333335), ((1025, '35-44'), 4.0), ((1039, '25-34'), 3.8333333333333335), ((1043, '45-54'), 4.375), ((1020, '35-44'), 3.8125), ((1025, '18-24'), 4.5), ((1013, '35-44'), 4.25)]


## Bài 05

In [7]:
# Map occupation ID -> name
occ_dict = occupation_parsed.collectAsMap()
broadcast_occ = sc.broadcast(occ_dict)

# Gán occupation cho rating
user_occ = users_parsed.map(lambda x: (x[0], x[3]))
user_rating = ratings_rdd.map(lambda x: (x[0], x[2]))
joined = user_rating.join(user_occ)
occ_rating = joined.map(lambda x: (broadcast_occ.value.get(x[1][1], "Unknown"), x[1][0]))

# Tính tổng và trung bình
sumcnt = occ_rating.mapValues(lambda r: (r,1)).reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
stats = sumcnt.mapValues(lambda x: (x[0]/x[1], x[1]))
print("Thống kê theo nghề:", stats.collect())

Thống kê theo nghề: [('Doctor', (3.6904761904761907, 21)), ('Designer', (4.0, 13)), ('Student', (4.0, 8)), ('Accountant', (3.5833333333333335, 6)), ('Programmer', (4.25, 10)), ('Engineer', (3.5555555555555554, 18)), ('Lawyer', (3.6470588235294117, 17)), ('Artist', (3.727272727272727, 11)), ('Consultant', (3.857142857142857, 14)), ('Journalist', (3.8529411764705883, 17)), ('Nurse', (3.8636363636363638, 11)), ('Manager', (3.46875, 16)), ('Teacher', (3.7, 5)), ('Salesperson', (3.6470588235294117, 17))]


## Bài 06

In [8]:
# Trích năm từ timestamp và tính trung bình theo năm
from datetime import datetime
year_rating = ratings_rdd.map(lambda x: (datetime.fromtimestamp(x[3]).year, (x[2], 1)))
sumcnt = year_rating.reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
stats = sumcnt.mapValues(lambda x: (x[0]/x[1], x[1]))
print("Thống kê theo năm:", sorted(stats.collect()))

Thống kê theo năm: [(2020, (3.7527173913043477, 184))]
